## **1: Importación de librerías y conexión a Google Drive**

### **1.1 Importación de librerías**

In [ ]:
import os
import re
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from google.colab import drive

# Preprocesamiento y construcción del modelo
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# Evaluación del modelo
from sklearn.metrics import (
  accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  average_precision_score,
  confusion_matrix,
  classification_report,
  ConfusionMatrixDisplay
)

# Configuración de reproducibilidad
RANDOM_STATE = 42

### **1.2 Conexión a Google Drive**

In [ ]:
drive.mount('/content/drive')

## **2: Cargar archivos CSV**

In [ ]:
RUTA_DRIVE = "/content/drive/MyDrive/Kipu FraudShield/datos"

for carpeta_actual, subcarpetas, archivos in os.walk(RUTA_DRIVE):
  csv_encontrados = [
    archivo for archivo in archivos
    if archivo.lower().endswith(".csv")
  ]

  for archivo in csv_encontrados:
    print(os.path.join(carpeta_actual, archivo))

print()
# Cargar los cuatro archivos CSV
datos = {
  "transacciones": pd.read_csv(os.path.join(RUTA_DRIVE, "transacciones.csv")),
  "usuarios": pd.read_csv(os.path.join(RUTA_DRIVE, "usuarios.csv")),
  "dispositivos": pd.read_csv(os.path.join(RUTA_DRIVE, "dispositivos.csv")),
  "lista_negra": pd.read_csv(os.path.join(RUTA_DRIVE, "lista_negra.csv"))
}

# Verificar que se cargaron correctamente
for nombre, df in datos.items():
  print(f"{nombre}: {df.shape[0]} filas y {df.shape[1]} columnas")


## **3: Preparación de copias para limpieza**

In [ ]:
# Conservar los datasets originales
datos_originales = {
  nombre: df.copy()
  for nombre, df in datos.items()
}

# Crear copias para aplicar la limpieza
datos_limpios = {
  nombre: df.copy()
  for nombre, df in datos.items()
}

print("Copias creadas correctamente")

for nombre, df in datos_limpios.items():
  print(f"{nombre}: {df.shape[0]} filas y {df.shape[1]} columnas")

## **4: Limpieza y normalización de datos**

### **4.1 Eliminación de duplicados exactos**

In [ ]:
print("=== VALIDACIÓN DE DUPLICADOS ===")

for nombre, df in datos_limpios.items():

    filas_antes = len(df)
    duplicados = int(df.duplicated().sum())

    datos_limpios[nombre] = df.drop_duplicates().copy()

    filas_despues = len(datos_limpios[nombre])

    # Verificar que la limpieza fue correcta
    assert filas_despues == filas_antes - duplicados
    assert not datos_limpios[nombre].duplicated().any()

    print(f"\nTabla: {nombre}")
    print(f"Registros iniciales: {filas_antes}")
    print(f"Duplicados eliminados: {duplicados}")
    print(f"Registros finales: {filas_despues}")
    print("Estado: OK")


### **4.2 Normalización de categorías**

In [ ]:
# 1. Limpiar textos
columnas = {
    "transacciones": ["canal", "resultado_autorizacion", "pais_transaccion"],
    "usuarios": ["segmento", "pais"],
    "dispositivos": ["tipo", "pais_declarado"],
    "lista_negra": ["tipo", "motivo"]
}

for tabla, nombres in columnas.items():
    for columna in nombres:
        datos_limpios[tabla][columna] = (
            datos_limpios[tabla][columna]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace("", pd.NA)
        )

# 2. Unificar segmentos de usuarios
datos_limpios["usuarios"]["segmento"] = (
    datos_limpios["usuarios"]["segmento"].replace({
        "individuo": "persona",
        "comercio pequeño": "comercio_pequeno",
        "comercio grande": "comercio_grande"
    })
)

# 3. Unificar países
equivalencias_paises = {
    "chile": "cl", "argentina": "ar",
    "peru": "pe", "perú": "pe",
    "brasil": "br", "brazil": "br",
    "colombia": "co", "venezuela": "ve",
    "usa": "us", "eeuu": "us",
    "estados unidos": "us",
    "rusia": "ru", "russia": "ru",
    "nigeria": "ng"
}

columnas_pais = {
    "transacciones": "pais_transaccion",
    "usuarios": "pais",
    "dispositivos": "pais_declarado"
}

for tabla, columna in columnas_pais.items():
    datos_limpios[tabla][columna] = (
        datos_limpios[tabla][columna].replace(equivalencias_paises)
    )

# 4. Verificar resultados de la limpieza
print("=== PAÍSES NORMALIZADOS ===")
resumen_paises = pd.DataFrame({
    tabla: datos_limpios[tabla][columna].value_counts()
    for tabla, columna in columnas_pais.items()
}).fillna(0).astype(int)
display(resumen_paises)

print("\n=== SEGMENTOS NORMALIZADOS ===")
print(datos_limpios["usuarios"]["segmento"].value_counts(dropna=False))

print("\n=== TIPOS DE DISPOSITIVO ===")
print(datos_limpios["dispositivos"]["tipo"].value_counts(dropna=False))

### **4.3 Limpieza y normalización de montos y monedas**

In [ ]:
tx = datos_limpios["transacciones"]

# 1. Conservar montos originales sin sobrescribirlos
if "monto_original" not in tx.columns:
    tx["monto_original"] = tx["monto"].copy()

# 2. Normalizar monedas
tx["moneda"] = (
    tx["moneda"].astype("string")
    .str.strip().str.upper().replace("", pd.NA)
)

# 3. Limpiar formatos de montos
texto = (
    tx["monto_original"].astype("string")
    .str.strip()
    .str.replace("$", "", regex=False)
)

# Corregir separadores de miles en CLP
clp_miles = (
    tx["moneda"].eq("CLP").fillna(False)
    & texto.str.fullmatch(r"^-?\d{1,3}\.\d{3}$").fillna(False)
)

texto.loc[clp_miles] = (
    texto.loc[clp_miles].str.replace(".", "", regex=False)
)

# Convertir comas decimales y montos numéricos
tx["monto"] = pd.to_numeric(
    texto.str.replace(",", ".", regex=False),
    errors="coerce"
).astype("Float64")

# 4. Marcar valores problemáticos
tx["moneda_desconocida"] = tx["moneda"].isna().astype(int)

original = (
    tx["monto_original"].astype("string")
    .str.strip().str.replace("$", "", regex=False)
)

tx["monto_ambiguo"] = (
    tx["moneda"].isna()
    & original.str.fullmatch(
        r"^-?\d{1,3}[.,]\d{3}$"
    ).fillna(False)
).astype(int)

tx["monto_negativo"] = (
    tx["monto"].lt(0).fillna(False).astype(int)
)

tx["monto_cero"] = (
    tx["monto"].eq(0).fillna(False).astype(int)
)

# 5. Mostrar resultados
print("=== RESUMEN DE LIMPIEZA ===")
print("Transacciones:", len(tx))
print("Montos nulos:", tx["monto"].isna().sum())
print("CLP corregidos:", clp_miles.sum())
print("Monedas desconocidas:", tx["moneda_desconocida"].sum())
print("Montos ambiguos:", tx["monto_ambiguo"].sum())
print("Montos negativos:", tx["monto_negativo"].sum())
print("Montos en cero:", tx["monto_cero"].sum())

print("\n=== DISTRIBUCIÓN DE MONEDAS ===")
print(tx["moneda"].value_counts(dropna=False))

print("\n=== EJEMPLOS DE CONVERSIÓN ===")
display(tx[["monto_original", "monto", "moneda"]].head(10))

### **4.4 Limpieza y normalización de fechas**


In [ ]:
# 1. Fechas de usuarios y lista negra
fechas_simples = {
    "usuarios": "fecha_registro",
    "lista_negra": "fecha_alta"
}

for tabla, columna in fechas_simples.items():
    df = datos_limpios[tabla]
    original = f"{columna}_original"

    if original not in df.columns:
        df[original] = df[columna].copy()

    df[columna] = pd.to_datetime(
        df[original].astype("string").str.strip(),
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

    print(f"\n=== {tabla.upper()} ===")
    print("Fechas convertidas:", df[columna].notna().sum())
    print("Fechas no convertidas:", df[columna].isna().sum())


# 2. Fechas de transacciones y dispositivos
fechas_timestamps = {
    "transacciones": "timestamp",
    "dispositivos": "primera_vez"
}

for tabla, columna in fechas_timestamps.items():
    df = datos_limpios[tabla]
    original = f"{columna}_original"

    if original not in df.columns:
        df[original] = df[columna].copy()

    texto = df[original].astype("string").str.strip()

    # Identificar fechas con zona horaria
    tiene_zona = texto.str.contains(
        r"(?:Z|[+-]\d{2}:\d{2})$",
        regex=True,
        na=False
    )

    df[f"{columna}_tiene_zona"] = tiene_zona

    # Convertir fechas con zona horaria a UTC
    df[f"{columna}_utc"] = pd.to_datetime(
        texto.where(tiene_zona),
        format="mixed",
        utc=True,
        errors="coerce"
    )

    # Convertir fechas sin zona, sin inventar una
    sin_zona = texto.where(~tiene_zona)
    formato_dia_primero = sin_zona.str.contains(
        "/", regex=False, na=False
    )

    fechas_locales = pd.Series(
        pd.NaT, index=df.index, dtype="datetime64[ns]"
    )

    fechas_locales.loc[formato_dia_primero] = pd.to_datetime(
        sin_zona.loc[formato_dia_primero],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

    fechas_locales.loc[~formato_dia_primero] = pd.to_datetime(
        sin_zona.loc[~formato_dia_primero],
        format="mixed",
        errors="coerce"
    )

    df[f"{columna}_sin_zona"] = fechas_locales

    # 3. Validar resultados
    validas = (
        df[f"{columna}_utc"].notna()
        | df[f"{columna}_sin_zona"].notna()
    )

    print(f"\n=== {tabla.upper()} ===")
    print("Registros:", len(df))
    print("Fechas con zona:", tiene_zona.sum())
    print("Fechas sin zona:", (~tiene_zona).sum())
    print("Fechas normalizadas:", validas.sum())
    print("Fechas sin convertir:", (~validas).sum())

    # Comprobar que ninguna fecha tenga dos representaciones
    fechas_duplicadas = (
        df[f"{columna}_utc"].notna()
        & df[f"{columna}_sin_zona"].notna()
    )

    print("Fechas con ambas representaciones:",
          fechas_duplicadas.sum())

    assert not fechas_duplicadas.any(), (
        f"Error: fechas duplicadas en {tabla}"
    )


### **4.5 Limpieza y validación de valores faltantes e inválidos**

In [ ]:
tx = datos_limpios["transacciones"]
usuarios = datos_limpios["usuarios"]

# 1. Normalizar variables booleanas
mapa_bool = {
    "1": 1, "1.0": 1, "si": 1, "sí": 1, "true": 1,
    "0": 0, "0.0": 0, "no": 0, "false": 0
}

booleanas = {
    "usuarios": ["telefono_verificado", "verificado"],
    "dispositivos": ["emulador"],
    "lista_negra": ["vigente"]
}

for tabla, columnas in booleanas.items():
    df = datos_limpios[tabla]
    for col in columnas:
        original = f"{col}_original"
        if original not in df.columns:
            df[original] = df[col].copy()

        df[col] = (
            df[original].astype("string")
            .str.strip().str.lower()
            .map(mapa_bool).astype("Int64")
        )

# 2. Tratar niveles KYC
if "nivel_kyc_original" not in usuarios.columns:
    usuarios["nivel_kyc_original"] = usuarios["nivel_kyc"].copy()

kyc = pd.to_numeric(usuarios["nivel_kyc_original"], errors="coerce")

usuarios["kyc_especial"] = kyc.eq(-1).fillna(False).astype(int)
usuarios["kyc_desconocido"] = kyc.isna().astype(int)
usuarios["nivel_kyc_limpio"] = (
    kyc.where(kyc.isin([0, 1, 2, 3])).astype("Int64")
)

# 3. Tratar intentos previos
col = "intentos_previos_1h"
if f"{col}_original" not in tx.columns:
    tx[f"{col}_original"] = tx[col].copy()

intentos = pd.to_numeric(tx[f"{col}_original"], errors="coerce")
tx["intentos_previos_desconocidos"] = intentos.isna().astype(int)
tx["intentos_previos_invalidos"] = (
    intentos.notna() &
    ((intentos < 0) | (intentos % 1 != 0))
).astype(int)

# 4. Validar coordenadas
for col in ["latitud", "longitud"]:
    if f"{col}_original" not in tx.columns:
        tx[f"{col}_original"] = tx[col].copy()

lat = pd.to_numeric(tx["latitud_original"], errors="coerce")
lon = pd.to_numeric(tx["longitud_original"], errors="coerce")

tx["coordenadas_faltantes"] = (lat.isna() | lon.isna()).astype(int)
tx["coordenadas_cero"] = (lat.eq(0) & lon.eq(0)).astype(int)
tx["coordenadas_fuera_rango"] = (
    lat.notna() & lon.notna() &
    (~lat.between(-90, 90) | ~lon.between(-180, 180))
).astype(int)

tx["coordenadas_sospechosas"] = (
    tx[["coordenadas_faltantes", "coordenadas_cero",
        "coordenadas_fuera_rango"]].any(axis=1).astype(int)
)

# 5. Identificar campos faltantes
identificadores = [
    "dispositivo_id", "comercio_id",
    "tarjeta_enmascarada", "ip"
]

for col in identificadores:
    tx[f"{col}_faltante"] = tx[col].isna().astype(int)

# 6. Tratar variables categóricas faltantes
categoricas = {
    "usuarios": ["email_dominio"],
    "dispositivos": ["sistema_operativo"],
    "lista_negra": ["agregado_por", "motivo"]
}

for tabla, columnas in categoricas.items():
    df = datos_limpios[tabla]
    for col in columnas:
        limpio = (
            df[col].astype("string")
            .str.strip().replace("", pd.NA)
        )
        df[f"{col}_faltante"] = limpio.isna().astype(int)
        df[f"{col}_limpio"] = limpio.fillna("desconocido")

# 7. Tratar historial de transacciones
col = "transacciones_previas"
if f"{col}_original" not in usuarios.columns:
    usuarios[f"{col}_original"] = usuarios[col].copy()

historial = pd.to_numeric(
    usuarios[f"{col}_original"], errors="coerce"
)

usuarios["transacciones_previas_desconocidas"] = (
    historial.isna().astype(int)
)
usuarios["transacciones_previas_invalidas"] = (
    historial.notna() &
    ((historial < 0) | (historial % 1 != 0))
).astype(int)

usuarios["transacciones_previas_limpio"] = historial.where(
    (historial >= 0) & (historial % 1 == 0)
).astype("Int64")

# 8. Validación de resultados
print("=== RESULTADO DE LA LIMPIEZA ===")

for tabla, df in datos_limpios.items():
    print(f"{tabla}: {len(df)} registros")

print("\n=== BOOLEANOS SIN VALOR VÁLIDO ===")
for tabla, columnas in booleanas.items():
    for col in columnas:
        print(f"{tabla}.{col}:",
              datos_limpios[tabla][col].isna().sum())

print("\n=== INDICADORES ===")
print("KYC especiales:", usuarios["kyc_especial"].sum())
print("KYC desconocidos:", usuarios["kyc_desconocido"].sum())
print("Intentos desconocidos:", tx["intentos_previos_desconocidos"].sum())
print("Intentos inválidos:", tx["intentos_previos_invalidos"].sum())
print("Coordenadas sospechosas:", tx["coordenadas_sospechosas"].sum())
print("Historial desconocido:",
      usuarios["transacciones_previas_desconocidas"].sum())
print("Historial inválido:",
      usuarios["transacciones_previas_invalidas"].sum())

print("\n=== CAMPOS FALTANTES ===")
for col in identificadores:
    print(f"{col}:", tx[f"{col}_faltante"].sum())

for tabla, columnas in categoricas.items():
    for col in columnas:
        print(f"{tabla}.{col}:",
              datos_limpios[tabla][f"{col}_faltante"].sum())

### **4.6 Limpieza y validación de etiquetas de fraude**

In [ ]:
tx = datos_limpios["transacciones"]

# 1. Conservar etiquetas originales
if "es_fraude_original" not in tx.columns:
    tx["es_fraude_original"] = tx["es_fraude"].copy()

# 2. Normalizar etiquetas
etiquetas = (
    tx["es_fraude_original"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace("", pd.NA)
)

mapa_fraude = {
    "0": 0, "0.0": 0, "no": 0,
    "1": 1, "1.0": 1, "si": 1, "sí": 1
}

tx["es_fraude_limpio"] = (
    etiquetas.map(mapa_fraude).astype("Int64")
)

# 3. Separar datos para entrenamiento
tx_etiquetadas = tx.loc[
    tx["es_fraude_limpio"].notna()
].copy()

tx_sin_etiqueta = tx.loc[
    tx["es_fraude_limpio"].isna()
].copy()

tx_etiquetadas["es_fraude_limpio"] = (
    tx_etiquetadas["es_fraude_limpio"].astype(int)
)

# 4. Identificar etiquetas no reconocidas
no_reconocidas = (
    etiquetas.notna()
    & tx["es_fraude_limpio"].isna()
)

# 5. Validar resultados
print("=== ETIQUETAS NORMALIZADAS ===")
print(tx["es_fraude_limpio"].value_counts(dropna=False))

print("\n=== DISTRIBUCIÓN DE CLASES ===")

total = len(tx_etiquetadas)

for valor, nombre in [(0, "No fraude"), (1, "Fraude")]:
    cantidad = (tx_etiquetadas["es_fraude_limpio"] == valor).sum()
    porcentaje = cantidad / total * 100 if total else 0
    print(f"{nombre}: {cantidad} ({porcentaje:.2f}%)")

print("\n=== SEPARACIÓN DE DATOS ===")
print("Transacciones etiquetadas:", len(tx_etiquetadas))
print("Transacciones sin etiqueta:", len(tx_sin_etiqueta))
print("Etiquetas no reconocidas:", no_reconocidas.sum())

print("\n=== VALIDACIÓN FINAL ===")
assert len(tx_etiquetadas) + len(tx_sin_etiqueta) == len(tx)
assert not no_reconocidas.any(), "Existen etiquetas no reconocidas"

print("Total conservado:", len(tx), "- OK")

# 6. Validar identificadores del conjunto etiquetado

ids_duplicados = int(
    tx_etiquetadas["transaccion_id"].duplicated().sum()
)

ids_faltantes = int(
    tx_etiquetadas["transaccion_id"].isna().sum()
)

usuarios_unicos = int(
    tx_etiquetadas["usuario_id"].nunique()
)

print("\n=== VALIDACIÓN DE IDENTIFICADORES ===")
print("Transacciones etiquetadas:", len(tx_etiquetadas))
print("IDs duplicados:", ids_duplicados)
print("IDs faltantes:", ids_faltantes)
print("Usuarios diferentes:", usuarios_unicos)

assert ids_duplicados == 0, "Existen IDs de transacción duplicados"
assert ids_faltantes == 0, "Existen transacciones sin identificador"

print("Estado: OK")

## **5. Integración de los datasets**

### **5.1 Integración de transacciones con usuarios**

In [ ]:
tx = datos_limpios["transacciones"]
usuarios = datos_limpios["usuarios"]

# Seleccionar características de usuarios relevantes
columnas_usuarios = [
    "usuario_id",
    "pais",
    "segmento",
    "fecha_registro",
    "edad_cuenta_dias",
    "nivel_kyc_limpio",
    "kyc_especial",
    "kyc_desconocido",
    "verificado",
    "telefono_verificado",
    "transacciones_previas_limpio",
    "transacciones_previas_desconocidas",
    "email_dominio_limpio"
]

# Integración mediante LEFT JOIN
datos_integrados = tx.merge(
    usuarios[columnas_usuarios],
    on="usuario_id",
    how="left",
    validate="many_to_one",
    indicator="_relacion_usuario"
)

# Validar resultados
print("=== INTEGRACIÓN TRANSACCIONES + USUARIOS ===")

print("Transacciones originales:", len(tx))
print("Registros integrados:", len(datos_integrados))

print("\nEstado de integración:")
print(datos_integrados["_relacion_usuario"].value_counts())

print("\nTransacciones duplicadas después del JOIN:")
print(datos_integrados["transaccion_id"].duplicated().sum())

print("\nEjemplos:")
print(
    datos_integrados[
        ["transaccion_id", "usuario_id",
         "segmento", "nivel_kyc_limpio",
         "_relacion_usuario"]
    ].head(10)
)

### **5.2 Integración de transacciones con dispositivos**

In [ ]:
dispositivos = datos_limpios["dispositivos"]

# Seleccionar características relevantes
columnas_dispositivos = [
    "dispositivo_id",
    "tipo",
    "sistema_operativo_limpio",
    "usuarios_distintos",
    "emulador",
    "pais_declarado",
    "primera_vez_utc",
    "primera_vez_sin_zona"
]

# Integrar dispositivos mediante LEFT JOIN
datos_integrados = datos_integrados.merge(
    dispositivos[columnas_dispositivos],
    on="dispositivo_id",
    how="left",
    validate="many_to_one",
    indicator="_relacion_dispositivo"
)

# Validar resultados
print("=== INTEGRACIÓN DE DISPOSITIVOS ===")

print("Registros integrados:", len(datos_integrados))

print("\nEstado de integración:")
print(
    datos_integrados["_relacion_dispositivo"]
    .value_counts()
)

print("\nTransacciones duplicadas:")
print(
    datos_integrados["transaccion_id"]
    .duplicated()
    .sum()
)

print("\nEjemplos:")
print(
    datos_integrados[
        [
            "transaccion_id",
            "dispositivo_id",
            "tipo",
            "emulador",
            "usuarios_distintos",
            "_relacion_dispositivo"
        ]
    ].head(10)
)

### **5.3 Preparación de la lista negra**

In [ ]:
lista_negra = datos_limpios["lista_negra"]

# Conservar el tipo original
lista_negra["tipo_original"] = lista_negra["tipo"].copy()

# Normalizar mayúsculas y espacios
lista_negra["tipo"] = (
    lista_negra["tipo"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "device": "dispositivo",
        "merchant": "comercio",
        "card": "tarjeta"
    })
)

print("=== TIPOS NORMALIZADOS ===")
print(lista_negra["tipo"].value_counts(dropna=False))

print("\nCantidad de categorías:")
print(lista_negra["tipo"].nunique(dropna=True))

print("\nDuplicados por tipo y valor:")
print(
    lista_negra.duplicated(
        subset=["tipo", "valor"],
        keep=False
    ).sum()
)

print("\nTotal de registros:", len(lista_negra))

### **5.4 Coincidencias históricas de la lista negra**

In [ ]:
tx = datos_integrados
ln = datos_limpios["lista_negra"]

relaciones = {
    "ip": "ip",
    "dispositivo": "dispositivo_id",
    "comercio": "comercio_id",
    "tarjeta": "tarjeta_enmascarada"
}

# Obtener fecha calendario sin asumir zona horaria
fecha_tx = tx["timestamp_utc"].dt.date.astype("string")

fecha_sin_zona = (
    tx["timestamp_sin_zona"].dt.date.astype("string")
)

fecha_tx = fecha_tx.fillna(fecha_sin_zona)

# Las fechas originales de transacción siguen disponibles
# para auditoría; esta comparación es conservadora por día.

for tipo, columna in relaciones.items():

    entradas = ln.loc[
        ln["tipo"] == tipo,
        ["valor", "fecha_alta"]
    ].copy()

    # Primera aparición de cada identificador
    primeras_fechas = (
        entradas.groupby("valor")["fecha_alta"]
        .min()
    )

    fecha_alta = (
        tx[columna]
        .astype("string")
        .str.strip()
        .map(primeras_fechas)
    )

    indicador = (
        fecha_alta.dt.date.astype("string") < fecha_tx
    ).fillna(False)

    tx[f"lista_negra_{tipo}_historica"] = (
        indicador.astype(int)
    )

    print(
        f"Coincidencias históricas ({tipo}):",
        tx[f"lista_negra_{tipo}_historica"].sum()
    )

# Indicador de cualquier coincidencia histórica
columnas_historicas = [
    f"lista_negra_{tipo}_historica"
    for tipo in relaciones
]

tx["lista_negra_coincidencia_historica"] = (
    tx[columnas_historicas].max(axis=1)
)

print(
    "\nTransacciones con alguna coincidencia histórica:",
    tx["lista_negra_coincidencia_historica"].sum()
)

print("Total de transacciones:", len(tx))

### **5.5 Validación de coincidencias históricas**

In [ ]:
tx = datos_integrados

columnas_historicas = [
    "lista_negra_ip_historica",
    "lista_negra_dispositivo_historica",
    "lista_negra_comercio_historica",
    "lista_negra_tarjeta_historica"
]

print("=== VALIDACIÓN DE INTEGRACIÓN ===")

print("Total de transacciones:", len(tx))

print(
    "IDs de transacción duplicados:",
    tx["transaccion_id"].duplicated().sum()
)

print(
    "Indicadores históricos nulos:",
    tx[columnas_historicas].isna().sum().sum()
)

print("\n=== COINCIDENCIAS POR TIPO ===")
print(tx[columnas_historicas].sum())

print("\n=== INDICADOR GENERAL ===")

print(
    tx["lista_negra_coincidencia_historica"]
    .value_counts(dropna=False)
)

# Comprobar consistencia del indicador general
indicador_calculado = tx[columnas_historicas].max(axis=1)

print(
    "\nInconsistencias del indicador general:",
    (
        indicador_calculado !=
        tx["lista_negra_coincidencia_historica"]
    ).sum()
)

## **6. Selección y preparación de variables para el entrenamiento**

In [ ]:
# Trabajar únicamente con transacciones etiquetadas
df_modelo = datos_integrados.loc[
    datos_integrados["es_fraude_limpio"].notna()
].copy()

# Variables numéricas candidatas
variables_numericas = [
    "monto_negativo",
    "monto_cero",
    "moneda_desconocida",
    "monto_ambiguo",
    "intentos_previos_1h",
    "intentos_previos_desconocidos",
    "edad_cuenta_dias",
    "nivel_kyc_limpio",
    "kyc_especial",
    "kyc_desconocido",
    "verificado",
    "telefono_verificado",
    "transacciones_previas_limpio",
    "transacciones_previas_desconocidas",
    "usuarios_distintos",
    "emulador",
    "coordenadas_sospechosas",
    "dispositivo_id_faltante",
    "comercio_id_faltante",
    "ip_faltante"
]

# Variables categóricas candidatas
variables_categoricas = [
    "moneda",
    "canal",
    "mcc",
    "pais_transaccion",
    "resultado_autorizacion",
    "segmento",
    "tipo",
    "pais_declarado",
    "email_dominio_limpio"
]

variables_modelo = (
    variables_numericas + variables_categoricas
)

print("=== SELECCIÓN DE VARIABLES ===")

print("Variables numéricas:", len(variables_numericas))
print("Variables categóricas:", len(variables_categoricas))

print("\nTotal de variables candidatas:", len(variables_modelo))

print("\nColumnas no encontradas:")
faltantes = [
    col for col in variables_modelo
    if col not in df_modelo.columns
]
print(faltantes)

print("\nRegistros disponibles para entrenamiento:",
      len(df_modelo))

print("\nDistribución de etiquetas:")
print(df_modelo["es_fraude_limpio"].value_counts())

### **6.1 Revisión de variables y prevención de fugas de información**

In [ ]:
# Variables que requieren validación temporal
variables_en_revision = [
    "resultado_autorizacion",
    "usuarios_distintos",
    "edad_cuenta_dias"
]

# Crear selección provisional sin esas variables
variables_seguras = [
    columna for columna in variables_modelo
    if columna not in variables_en_revision
]

# Separar variables numéricas y categóricas
numericas_seguras = [
    columna for columna in variables_numericas
    if columna in variables_seguras
]

categoricas_seguras = [
    columna for columna in variables_categoricas
    if columna in variables_seguras
]

print("=== REVISIÓN DE VARIABLES ===")

print("Variables excluidas provisionalmente:")
print(variables_en_revision)

print("\nVariables numéricas:", len(numericas_seguras))
print("Variables categóricas:", len(categoricas_seguras))
print("Total de variables seleccionadas:", len(variables_seguras))

print("\n=== TIPOS DE DATOS ===")
print(df_modelo[variables_seguras].dtypes)

print("\n=== VALORES FALTANTES ===")
print(
    df_modelo[variables_seguras]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

### **6.2 Preparación de variables X e y**

In [ ]:
# Crear copias para no modificar las listas originales
numericas_finales = numericas_seguras.copy()
categoricas_finales = categoricas_seguras.copy()

# MCC es un código categórico, no una cantidad
if "mcc" in numericas_finales:
    numericas_finales.remove("mcc")

if "mcc" not in categoricas_finales:
    categoricas_finales.append("mcc")

# Crear matriz de características
X = df_modelo[
    numericas_finales + categoricas_finales
].copy()

# Crear variable objetivo
y = df_modelo["es_fraude_limpio"].astype(int)

# Convertir categóricas a texto
for columna in categoricas_finales:
    X[columna] = X[columna].astype("string")

# Convertir numéricas a float para Scikit-learn
for columna in numericas_finales:
    X[columna] = pd.to_numeric(
        X[columna],
        errors="coerce"
    ).astype(float)

print("=== PREPARACIÓN DE VARIABLES ===")

print("Variables numéricas:", len(numericas_finales))
print("Variables categóricas:", len(categoricas_finales))
print("Total de variables:", X.shape[1])

print("\nDimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

print("\nDistribución de clases:")
print(y.value_counts())

print("\nValores infinitos en numéricas:")
print(
    np.isinf(X[numericas_finales].to_numpy()).sum()
)

print("\nValores nulos totales:")
print(X.isna().sum().sum())

### **6.3 División de entrenamiento y prueba por usuario**

In [ ]:
# Identificar los usuarios de las transacciones etiquetadas
grupos = df_modelo["usuario_id"].astype("string")

# División con 80% entrenamiento y 20% prueba
divisor = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

indice_train, indice_test = next(
    divisor.split(X, y, groups=grupos)
)

X_train = X.iloc[indice_train].copy()
X_test = X.iloc[indice_test].copy()

y_train = y.iloc[indice_train].copy()
y_test = y.iloc[indice_test].copy()

grupos_train = grupos.iloc[indice_train]
grupos_test = grupos.iloc[indice_test]

print("=== DIVISIÓN DE DATOS ===")

print("Registros de entrenamiento:", len(X_train))
print("Registros de prueba:", len(X_test))

print("\nDistribución en entrenamiento:")
print(y_train.value_counts())

print("\nDistribución en prueba:")
print(y_test.value_counts())

print("\nUsuarios presentes en ambos conjuntos:")
print(
    len(
        set(grupos_train.dropna())
        & set(grupos_test.dropna())
    )
)

print("\nPorcentaje de fraude en entrenamiento:")
print(round(y_train.mean() * 100, 2), "%")

print("\nPorcentaje de fraude en prueba:")
print(round(y_test.mean() * 100, 2), "%")

### **6.4 Preprocesamiento e imputación de valores faltantes**

In [ ]:
# Pipeline para variables numéricas
pipeline_numerico = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="median"))
])

# Pipeline para variables categóricas
pipeline_categorico = Pipeline(steps=[
    ("imputador", SimpleImputer(
        strategy="constant",
        fill_value="desconocido"
    )),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combinar ambos pipelines
preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", pipeline_numerico, numericas_finales),
        ("categoricas", pipeline_categorico, categoricas_finales)
    ]
)

# Crear copias compatibles con Scikit-learn
X_train_preparado = X_train.copy()
X_test_preparado = X_test.copy()

for columna in categoricas_finales:
    X_train_preparado[columna] = (
        X_train_preparado[columna]
        .astype("object")
        .where(X_train_preparado[columna].notna(), np.nan)
    )

    X_test_preparado[columna] = (
        X_test_preparado[columna]
        .astype("object")
        .where(X_test_preparado[columna].notna(), np.nan)
    )

# Ajustar únicamente con entrenamiento
X_train_transformado = preprocesador.fit_transform(
    X_train_preparado
)

# Transformar prueba sin volver a ajustar
X_test_transformado = preprocesador.transform(
    X_test_preparado
)

print("=== PREPROCESAMIENTO ===")

print("Entrenamiento transformado:",
      X_train_transformado.shape)

print("Prueba transformada:",
      X_test_transformado.shape)

print("\nValores nulos en entrenamiento:",
      np.isnan(X_train_transformado.toarray()
               if hasattr(X_train_transformado, "toarray")
               else X_train_transformado).sum())

print("Valores nulos en prueba:",
      np.isnan(X_test_transformado.toarray()
               if hasattr(X_test_transformado, "toarray")
               else X_test_transformado).sum())

### **6.5 Entrenamiento del modelo Random Forest**

In [ ]:
# Crear pipeline completo
modelo_fraude = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("clasificador", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Entrenar utilizando únicamente el conjunto de entrenamiento
modelo_fraude.fit(X_train_preparado, y_train)

print("=== ENTRENAMIENTO RANDOM FOREST ===")
print("Modelo entrenado correctamente.")

print("\nAlgoritmo:")
print(modelo_fraude.named_steps["clasificador"])

print("\nRegistros utilizados:", len(X_train_preparado))
print("Variables originales:", X_train_preparado.shape[1])

print("\nClases aprendidas:")
print(modelo_fraude.classes_)

### **6.6 Evaluación del modelo Random Forest**

In [ ]:
# Predicciones sobre el conjunto de prueba
y_pred = modelo_fraude.predict(X_test_preparado)

# Probabilidad estimada de fraude
y_prob = modelo_fraude.predict_proba(X_test_preparado)[:, 1]

# Calcular métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
pr_auc = average_precision_score(y_test, y_prob)

print("=== EVALUACIÓN RANDOM FOREST ===")

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"PR-AUC (Average Precision): {pr_auc:.4f}")

print("\n=== MATRIZ DE CONFUSIÓN ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== REPORTE DE CLASIFICACIÓN ===")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["No fraude", "Fraude"],
        zero_division=0
    )
)

### **6.7 Importancia de las variables del modelo**

In [ ]:
# Obtener el preprocesador y el clasificador
prepro = modelo_fraude.named_steps["preprocesamiento"]
clasificador = modelo_fraude.named_steps["clasificador"]

# Obtener nombres de variables después del One-Hot Encoding
nombres_variables = prepro.get_feature_names_out()

# Obtener importancia de las variables
importancias = pd.DataFrame({
    "Variable": nombres_variables,
    "Importancia": clasificador.feature_importances_
})

# Ordenar de mayor a menor
importancias = importancias.sort_values(
    "Importancia",
    ascending=False
)

print("=== 15 VARIABLES MÁS IMPORTANTES ===")
print(importancias.head(15).to_string(index=False))

### **6.8 Comparación del modelo con y sin intentos previos**

In [ ]:
# Excluir intentos previos y su indicador de ausencia
columnas_excluir = [
    "intentos_previos_1h",
    "intentos_previos_desconocidos"
]

numericas_reducidas = [
    col for col in numericas_finales
    if col not in columnas_excluir
]

# Crear preprocesador reducido
preprocesador_reducido = ColumnTransformer(
    transformers=[
        (
            "numericas",
            clone(pipeline_numerico),
            numericas_reducidas
        ),
        (
            "categoricas",
            clone(pipeline_categorico),
            categoricas_finales
        )
    ]
)

# Crear segundo modelo Random Forest
modelo_sin_intentos = Pipeline(steps=[
    ("preprocesamiento", preprocesador_reducido),
    ("clasificador", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Seleccionar variables restantes
columnas_reducidas = (
    numericas_reducidas + categoricas_finales
)

# Entrenar el segundo modelo
modelo_sin_intentos.fit(
    X_train_preparado[columnas_reducidas],
    y_train
)

# Obtener predicciones
pred_sin = modelo_sin_intentos.predict(
    X_test_preparado[columnas_reducidas]
)

prob_sin = modelo_sin_intentos.predict_proba(
    X_test_preparado[columnas_reducidas]
)[:, 1]

# Comparar métricas
comparacion = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Average Precision"
    ],
    "Modelo original": [
        accuracy,
        precision,
        recall,
        f1,
        pr_auc
    ],
    "Modelo sin intentos": [
        accuracy_score(y_test, pred_sin),
        precision_score(y_test, pred_sin, zero_division=0),
        recall_score(y_test, pred_sin, zero_division=0),
        f1_score(y_test, pred_sin, zero_division=0),
        average_precision_score(y_test, prob_sin)
    ]
})

print("=== COMPARACIÓN DE MODELOS ===")
print(comparacion.round(4).to_string(index=False))

print("\n=== MATRIZ DE CONFUSIÓN SIN INTENTOS ===")
print(confusion_matrix(y_test, pred_sin))


### **6.9 Selección provisional del modelo Random Forest**

In [ ]:
# Conservar el modelo con mejores resultados experimentales
modelo_seleccionado = modelo_fraude

# Guardar las columnas utilizadas durante el entrenamiento
variables_seleccionadas = X_train_preparado.columns.tolist()

# Variables que no deben utilizarse por fuga de información
variables_prohibidas = [
    "es_fraude",
    "es_fraude_limpio",
    "revisado_por_analista",
    "motivo_bloqueo",
    "resultado_autorizacion"
]

# Comprobar que no estén presentes
prohibidas_en_modelo = [
    col for col in variables_seleccionadas
    if col in variables_prohibidas
]

print("=== MODELO SELECCIONADO ===")

print("Algoritmo: Random Forest")
print("Variables utilizadas:", len(variables_seleccionadas))

print("\nVariables prohibidas encontradas:")
print(prohibidas_en_modelo)

print("\nResultados experimentales:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Average Precision: {pr_auc:.4f}")

print("\nEstado: CANDIDATO EXPERIMENTAL")
print("Pendiente: validación de disponibilidad temporal")

## **7. Validación temporal del modelo**

In [ ]:
df = df_modelo.copy()

print("=== VALIDACIÓN TEMPORAL ===")

print("Transacciones etiquetadas:", len(df))

# Obtener fecha calendario de la transacción
fecha_tx = df["timestamp_utc"].dt.tz_localize(None)

fecha_tx = fecha_tx.fillna(
    df["timestamp_sin_zona"]
)

# Comparar fechas de registro de usuarios
fecha_usuario = pd.to_datetime(
    df["fecha_registro"],
    errors="coerce"
)

# Margen de un día para evitar falsas alertas
# por diferencias de zona horaria
usuarios_posteriores = (
    fecha_usuario.dt.normalize()
    > fecha_tx.dt.normalize() + pd.Timedelta(days=1)
)

print("\nUsuarios registrados después de la transacción:")
print(usuarios_posteriores.sum())

# Comparar dispositivos solo cuando ambas fechas
# tienen zona horaria conocida
fechas_comparables = (
    df["timestamp_utc"].notna()
    & df["primera_vez_utc"].notna()
)

dispositivos_posteriores = (
    fechas_comparables
    & (
        df["primera_vez_utc"]
        > df["timestamp_utc"] + pd.Timedelta(days=1)
    )
)

print("\nTransacciones con fechas de dispositivo comparables:")
print(fechas_comparables.sum())

print("Dispositivos registrados después de la transacción:")
print(dispositivos_posteriores.sum())

print("\n=== VARIABLES SENSIBLES A REVISIÓN ===")

variables_revisar = [
    "intentos_previos_1h",
    "transacciones_previas_limpio",
    "nivel_kyc_limpio",
    "verificado",
    "emulador"
]

for variable in variables_revisar:
    print(
        f"{variable}:",
        "PRESENTE" if variable in variables_seleccionadas
        else "NO UTILIZADA"
    )

print("\nModelo seleccionado:", type(
    modelo_seleccionado.named_steps["clasificador"]
).__name__)

### **7.1 Investigación de inconsistencias temporales**

In [ ]:
df = df_modelo.copy()

# Fecha de transacción, conservando la referencia disponible
fecha_tx = (
    df["timestamp_utc"]
    .dt.tz_localize(None)
    .fillna(df["timestamp_sin_zona"])
)

# Usuarios con registro posterior por más de un día
usuarios_posteriores = (
    df["fecha_registro"].notna()
    & (
        df["fecha_registro"].dt.normalize()
        > fecha_tx.dt.normalize() + pd.Timedelta(days=1)
    )
)

# Dispositivos con primera aparición posterior
# Solo comparar fechas con zona horaria conocida
dispositivos_posteriores = (
    df["timestamp_utc"].notna()
    & df["primera_vez_utc"].notna()
    & (
        df["primera_vez_utc"]
        > df["timestamp_utc"] + pd.Timedelta(days=1)
    )
)

print("=== INCONSISTENCIAS TEMPORALES ===")

print("Usuarios posteriores:", usuarios_posteriores.sum())
print("Dispositivos posteriores:", dispositivos_posteriores.sum())

print(
    "Transacciones con alguna inconsistencia:",
    (usuarios_posteriores | dispositivos_posteriores).sum()
)

print("\n=== ETIQUETAS DE LOS CASOS ===")

print("Usuarios posteriores:")
print(
    df.loc[usuarios_posteriores, "es_fraude_limpio"]
    .value_counts(dropna=False)
)

print("\nDispositivos posteriores:")
print(
    df.loc[dispositivos_posteriores, "es_fraude_limpio"]
    .value_counts(dropna=False)
)

print("\n=== EJEMPLOS DE USUARIOS ===")

print(
    df.loc[
        usuarios_posteriores,
        ["transaccion_id", "usuario_id",
         "timestamp_original", "fecha_registro",
         "es_fraude_limpio"]
    ].head(5).to_string(index=False)
)

print("\n=== EJEMPLOS DE DISPOSITIVOS ===")

print(
    df.loc[
        dispositivos_posteriores,
        ["transaccion_id", "dispositivo_id",
         "timestamp_utc", "primera_vez_utc",
         "es_fraude_limpio"]
    ].head(5).to_string(index=False)
)

### **7.2 Entrenamiento y evaluación del modelo conservador**

In [ ]:
# Variables numéricas de la transacción
numericas_conservadoras = [
    "monto_negativo",
    "monto_cero",
    "moneda_desconocida",
    "monto_ambiguo",
    "coordenadas_sospechosas",
    "dispositivo_id_faltante",
    "comercio_id_faltante",
    "ip_faltante"
]

# Variables categóricas de la transacción
categoricas_conservadoras = [
    "moneda",
    "canal",
    "mcc",
    "pais_transaccion"
]

columnas_conservadoras = (
    numericas_conservadoras + categoricas_conservadoras
)

# Preprocesamiento independiente
preprocesador_conservador = ColumnTransformer(
    transformers=[
        ("numericas",
         clone(pipeline_numerico),
         numericas_conservadoras),

        ("categoricas",
         clone(pipeline_categorico),
         categoricas_conservadoras)
    ]
)

# Modelo conservador
modelo_conservador = Pipeline(steps=[
    ("preprocesamiento", preprocesador_conservador),
    ("clasificador", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Entrenar con los mismos grupos de entrenamiento
modelo_conservador.fit(
    X_train_preparado[columnas_conservadoras],
    y_train
)

# Evaluar con el mismo conjunto de prueba
pred_conservador = modelo_conservador.predict(
    X_test_preparado[columnas_conservadoras]
)

prob_conservador = modelo_conservador.predict_proba(
    X_test_preparado[columnas_conservadoras]
)[:, 1]

print("=== MODELO CONSERVADOR ===")

print("Variables utilizadas:", len(columnas_conservadoras))

print("Accuracy:",
      round(accuracy_score(y_test, pred_conservador), 4))

print("Precision:",
      round(precision_score(
          y_test, pred_conservador, zero_division=0), 4))

print("Recall:",
      round(recall_score(
          y_test, pred_conservador, zero_division=0), 4))

print("F1-score:",
      round(f1_score(
          y_test, pred_conservador, zero_division=0), 4))

print("Average Precision:",
      round(average_precision_score(
          y_test, prob_conservador), 4))

print("\n=== MATRIZ DE CONFUSIÓN ===")
print(confusion_matrix(y_test, pred_conservador))

## **8. Exportación del modelo conservador**

In [ ]:
# Carpeta de destino en Google Drive
ruta_modelos = "/content/drive/MyDrive/Kipu FraudShield/modelos"

os.makedirs(ruta_modelos, exist_ok=True)

# Seleccionar modelo conservador
modelo_para_exportar = modelo_conservador

# Ruta del archivo
ruta_modelo = os.path.join(
    ruta_modelos,
    "kipu_fraudshield_ep3_conservador.joblib"
)

# Guardar pipeline completo
joblib.dump(modelo_para_exportar, ruta_modelo)

# Guardar información de las variables necesarias
metadata = {
    "nombre": "Kipu FraudShield",
    "version": "ep3_prototipo",
    "algoritmo": "RandomForestClassifier",
    "variables": columnas_conservadoras,
    "numericas": numericas_conservadoras,
    "categoricas": categoricas_conservadoras,
    "estado": "experimental",
    "limitaciones": (
        "Modelo entrenado con una muestra etiquetada reducida. "
        "Pendiente validacion temporal completa e independiente."
    )
}

ruta_metadata = os.path.join(
    ruta_modelos,
    "kipu_fraudshield_ep3_metadata.json"
)

with open(ruta_metadata, "w", encoding="utf-8") as archivo:
    json.dump(metadata, archivo, indent=4, ensure_ascii=False)

print("=== EXPORTACIÓN COMPLETADA ===")

print("Modelo guardado en:")
print(ruta_modelo)

print("\nMetadata guardada en:")
print(ruta_metadata)

print("\nArchivos existentes:")
print(os.listdir(ruta_modelos))

print("\nTamaño del modelo (MB):")
print(round(os.path.getsize(ruta_modelo) / (1024**2), 2))

### **8.1 Verificación del modelo exportado**

In [ ]:
# Cargar modelo desde Google Drive
modelo_cargado = joblib.load(ruta_modelo)

# Cargar metadata
with open(ruta_metadata, "r", encoding="utf-8") as archivo:
    metadata_cargada = json.load(archivo)

print("=== VERIFICACIÓN DEL MODELO ===")

print("Modelo cargado correctamente.")
print("Nombre:", metadata_cargada["nombre"])
print("Versión:", metadata_cargada["version"])
print("Estado:", metadata_cargada["estado"])

print("\nVariables requeridas:")
print(len(metadata_cargada["variables"]))

# Seleccionar ejemplos del conjunto de prueba
ejemplos = X_test_preparado[
    metadata_cargada["variables"]
].head(5).copy()

# Obtener predicciones
predicciones = modelo_cargado.predict(ejemplos)

# Obtener probabilidades estimadas de fraude
probabilidades = modelo_cargado.predict_proba(ejemplos)[:, 1]

# Comprobar que la versión cargada y la original coinciden
probabilidades_originales = modelo_para_exportar.predict_proba(
    ejemplos
)[:, 1]

np.testing.assert_allclose(
    probabilidades,
    probabilidades_originales
)

print("\n=== PREDICCIONES DE PRUEBA ===")

resultados = pd.DataFrame({
    "Prediccion": predicciones,
    "Probabilidad_fraude": probabilidades.round(4)
})

print(resultados.to_string(index=False))

print("\nVerificación: predicciones idénticas al modelo original.")
print("Modelo listo para integrarse en el prototipo.")